In [63]:
import os, json, glob, math
from pathlib import Path

import cv2
import numpy as np
import layoutparser as lp
from sklearn.cluster import KMeans
from layoutparser.ocr import TesseractAgent
from skimage.transform import rotate as sk_rotate


In [125]:
from lxml import etree
from shapely.geometry import box
from shapely.strtree import STRtree

## ParseLayout

In [117]:
def debug_draw_layout(image_rgb, blocks, gutters=None, path="debug.png"):
    img = image_rgb.copy()
    for b in blocks:
        x1,y1,x2,y2 = map(int, [b.block.x_1, b.block.y_1, b.block.x_2, b.block.y_2])
        color = (0,255,0) if b.type in ("Text","Title") else (255,0,0)
        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
    if gutters:
        for x in gutters:
            cv2.line(img, (int(x),0), (int(x),img.shape[0]-1), (0,0,255), 1)
    cv2.imwrite(path, img[..., ::-1])  # RGB->BGR for write

def deskew_grayscale(img_rgb):
    # to gray
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    # binary (adaptive works well on microfilm/uneven pages)
    bw = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY, 35, 15)
    # estimate skew from Hough lines (fallback to 0)
    edges = cv2.Canny(bw, 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)  # high threshold to avoid noise
    angle_deg = 0.0
    if lines is not None:
        # compute dominant angle around 0 or 180; convert from radians
        angles = []
        for rho, theta in lines[:,0]:
            a = (theta * 180.0 / np.pi) - 90  # map to [-90,90]
            if -30 < a < 30:
                angles.append(a)
        if angles:
            angle_deg = float(np.median(angles))
    if abs(angle_deg) > 0.1:
        img_rgb = (sk_rotate(img_rgb, angle_deg, resize=False, mode="edge") * 255).astype(np.uint8)
    # light contrast boost for faint ink
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    g = cv2.equalizeHist(g)
    img_rgb = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)
    return img_rgb

def reclassify_by_text_density(image_rgb, blocks, ocr_agent, lang="spa+eng",
                               min_chars=30, alpha_ratio=0.6):
    """
    For any 'Figure' block: OCR a quick crop; if it contains enough letters
    (mostly alphabetic, likely text), flip type to 'Text'.
    """
    fixed = []
    for b in blocks:
        if b.type == "Figure":
            seg = b.pad(5,5,5,5).crop_image(image_rgb)
            txt = ocr_agent.detect(seg)
            s = (txt or "").strip()
            letters = sum(ch.isalpha() for ch in s)
            if len(s) >= min_chars and (letters / max(1, len(s))) >= alpha_ratio:
                b.type = "Text"
        fixed.append(b)
    return lp.Layout(fixed)

def group_into_columns(blocks, page_width):
    """
    Heuristic: cluster blocks by x-center into K columns (K chosen by elbow rule up to MAX_COLS),
    then sort columns left->right and blocks top->bottom within each column.
    """
    if not blocks:
        return []

    xs = np.array([[b.block.x_1 + b.block.x_2] for b in blocks]) / 2.0

    best_k, best_inertia = 1, float("inf")
    for k in range(1, min(MAX_COLS, len(blocks)) + 1):
        km = KMeans(n_clusters=k, n_init="auto", random_state=0).fit(xs)
        if km.inertia_ < best_inertia * 0.75:  # elbow-ish drop
            best_k, best_inertia = k, km.inertia_
    km = KMeans(n_clusters=best_k, n_init="auto", random_state=0).fit(xs)
    labels = km.labels_

    # Order columns by cluster center x
    centers = [xs[labels == li].mean() for li in range(best_k)]
    col_order = np.argsort(centers)

    ordered_blocks = []
    for col_idx in col_order:
        col_blocks = [b for b, li in zip(blocks, labels) if li == col_idx]
        col_blocks.sort(key=lambda b: b.block.y_1)  # top->bottom
        ordered_blocks.extend(col_blocks)
    return ordered_blocks


def extract_text_from_blocks(image_rgb, blocks):
    texts = []
    for b in blocks:
        # light padding can improve OCR robustness
        segment = b.pad(left=5, right=5, top=5, bottom=5).crop_image(image_rgb)
        # Use Tesseract; croppings are paragraph-ish -> PSM 6 helps
        text = ocr_agent.detect(segment, return_response=True)
        # .detect returns a string by default; with return_response=True you can get more info
        if isinstance(text, dict) and "text" in text:
            text = text["text"]
        texts.append((b, text.strip()))
    return texts

In [90]:
IMAGES_GLOB = "data/newspapers/*.jpg"
OUT_TXT_DIR   = Path("outputs/txt")
OUT_JSON_DIR  = Path("outputs/json")
OUT_TXT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

LANGS = "spa+eng"              # Spanish + English fallback
SCORE_THRESH = 0.01            # filter weaker layout detections
MAX_COLS = 6                   # max columns expected in a page


In [ ]:
# model = lp.Detectron2LayoutModel(
#     # config_path="lp://PubLayNet/faster_rcnn_R_50_FPN_3x/config",
#     config_path="lp://PubLayNet/mask_rcnn_X_101_32x8d_FPN_3x/config",
#     model_path=os.path.expanduser("~/.torch/iopath_cache/s/dgy9c10wykk4lq4/model_final.pth"),
#     label_map={0: "Text", 1: "Title", 2: "List", 3: "Table", 4: "Figure"},
#     extra_config=["MODEL.ROI_HEADS.SCORE_THRESH_TEST", SCORE_THRESH],
#     device="cpu",          # force CPU on Mac
# )

In [98]:
model = lp.Detectron2LayoutModel(
    config_path="lp://PubLayNet/faster_rcnn_R_50_FPN_3x/config",
    # config_path="lp://NewspaperNavigator/faster_rcnn_R_50_FPN_3x/config",
    # config_path="lp://PrimaLayout/mask_rcnn_R_50_FPN_3x/config",
    model_path=os.path.expanduser("~/.torch/iopath_cache/s/dgy9c10wykk4lq4/model_final.pth"),
    # model_path=os.path.expanduser("~/.torch/iopath_cache/s/h7th27jfv19rxiy/model_final.pth"),
    label_map={0: "Text", 1: "Title", 2: "List", 3: "Table", 4: "Figure"},
    extra_config=[
        "INPUT.MIN_SIZE_TEST", 1600,         # 800 -> 1600/2000 helps small blocks
        "INPUT.MAX_SIZE_TEST", 2400,         # allow bigger long edge
        "MODEL.ROI_HEADS.SCORE_THRESH_TEST", 0.4,   # 0.8 -> 0.4 finds faint blocks
        "MODEL.RPN.PRE_NMS_TOPK_TEST", 2000,        # more proposals
        "MODEL.RPN.POST_NMS_TOPK_TEST", 1000
    ],
    device="cpu",          # force CPU on Mac
)

In [4]:
ocr_agent = TesseractAgent(languages=LANGS)  # uses the 'tesseract' binary


## ALTO

In [157]:
# ---------- 1) Parse ALTO lines ----------
def parse_alto_textlines(alto_path):
    """
    Returns:
      lines: list of dicts:
        {
          'geom': shapely Polygon,
          'x','y','w','h': floats,
          'strings': [ {'x','y','w','h','text','text_norm','subs_type'} ... ],
          'text': str  # joined text for the line (with spaces, de-hyphenation hints respected)
        }
    """
    tree = etree.parse(alto_path)
    root = tree.getroot()

    # Robust namespace handling for ALTO v2/v3
    nsmap = {k or 'alto': v for k, v in root.nsmap.items() if v}
    ns = {'a': nsmap.get('alto', root.tag.split('}')[0].strip('{'))}

    def xp(node, path):
        return node.xpath(path, namespaces=ns)

    lines = []
    # Walk structure: TextBlock -> TextLine -> (String|SP)
    for tline in xp(root, './/a:TextLine'):
        # line geometry
        try:
            lx = float(tline.get('HPOS')); ly = float(tline.get('VPOS'))
            lw = float(tline.get('WIDTH')); lh = float(tline.get('HEIGHT'))
        except (TypeError, ValueError):
            # Some ALTOs keep geometry only on Strings; skip if line lacks it
            # or compute from children if you prefer.
            strings = xp(tline, './a:String')
            if not strings:
                continue
            xs = [float(s.get('HPOS')) for s in strings]
            ys = [float(s.get('VPOS')) for s in strings]
            ws = [float(s.get('WIDTH')) for s in strings]
            hs = [float(s.get('HEIGHT')) for s in strings]
            lx, ly = min(xs), min(ys)
            lw = max(x+ w for x, w in zip(xs, ws)) - lx
            lh = max(y+ h for y, h in zip(ys, hs)) - ly

        geom = box(lx, ly, lx + lw, ly + lh)

        # Collect tokens and spaces in document order
        tokens = []
        line_fragments = []
        for child in tline:
            tag = child.tag.split('}')[-1]
            if tag == 'String':
                x = float(child.get('HPOS')); y = float(child.get('VPOS'))
                w = float(child.get('WIDTH')); h = float(child.get('HEIGHT'))
                raw = child.get('CONTENT', '') or ''
                subs_type = child.get('SUBS_TYPE')  # e.g., HypPart1/HypPart2
                subs_cont = child.get('SUBS_CONTENT')  # de-hyphenated form for this token
                text_norm = subs_cont if subs_cont else raw

                tokens.append({
                    'x': x, 'y': y, 'w': w, 'h': h,
                    'text': raw, 'text_norm': text_norm, 'subs_type': subs_type
                })
                line_fragments.append(text_norm)
            elif tag == 'SP':
                # Represent explicit spaces; many corpora omit SPs entirely,
                # but when present they’re the most faithful indicator.
                line_fragments.append(' ')

        # Fallback: if no SPs, put single spaces between tokens
        if ' ' not in ''.join(line_fragments):  # naive check
            line_text = ' '.join(tok['text_norm'] for tok in tokens)
        else:
            # Already contains spaces from <SP>, but ensure no double-spacing
            line_text = ' '.join(''.join(line_fragments).split())

        # Handle simple hyphenation at line ends if SUBS markers exist:
        # If last token is HypPart1, keep its SUBS_CONTENT (already in text_norm)
        # and do not add extra hyphen; the next line will carry the continuation.
        # (We already used text_norm, so nothing more to do here.)

        lines.append({
            'geom': geom,
            'x': lx, 'y': ly, 'w': lw, 'h': lh,
            'strings': tokens,
            'text': line_text
        })

    return lines

# ---------- 2) Index lines ----------
# def build_line_index(lines):
#     geoms = [ln['geom'] for ln in lines]
#     index = STRtree(geoms)
#     g2rec = {ln['geom']: ln for ln in lines}
#     return index, g2rec

def build_line_index(lines):
    geoms = [ln['geom'] for ln in lines]
    index = STRtree(geoms)
    # Instead of geom→line, keep a parallel list
    return index, geoms, lines

# ---------- 3) Extract text for an LP region using TextLine ----------
def text_for_lp_region_lines(lp_bbox, line_index, geoms, lines, containment='intersect'):
    region = box(*lp_bbox)
    idxs = line_index.query(region)  # returns indices in Shapely 2.x

    selected = []
    for i in idxs:
        g = geoms[i]
        if containment == 'inside':
            if region.contains(g):
                selected.append(lines[i])
        else:
            if region.intersects(g):
                selected.append(lines[i])

    if not selected:
        return "", []

    # Order by top→bottom, then left→right
    selected.sort(key=lambda ln: (ln['y'], ln['x']))
    text = '\n'.join(ln['text'] for ln in selected)
    return text, selected

# ---------- 4) Wiring with LayoutParser ----------
def lp_elements_to_bboxes(lp_layout):
    """
    Accepts a layoutparser.Layout or list of elements.
    Returns list of dicts: {'type': str|None, 'bbox': (x1,y1,x2,y2), 'score': float|None, 'id': int}
    """
    out = []
    for i, el in enumerate(lp_layout):
        # Robustly get bbox
        try:
            x1, y1, x2, y2 = el.block.x_1, el.block.y_1, el.block.x_2, el.block.y_2
        except AttributeError:
            # Try other accessors or assume el is already a bbox tuple
            x1, y1, x2, y2 = el  # last resort
        out.append({
            'id': i,
            'type': getattr(el, 'type', None),
            'score': getattr(el, 'score', None),
            'bbox': (float(x1), float(y1), float(x2), float(y2)),
        })
    return out

## RUN

In [50]:
glob.glob(IMAGES_GLOB)

['data/newspapers_low/sample_1.jpg']

In [78]:
img_paths = sorted(glob.glob(IMAGES_GLOB))
ip = img_paths[1]

In [58]:
ip

'data/newspapers/service-ndnp-prru-batch_prru_ballena_ver01-data-sn90070270-00271761995-1899021301-0345.jpg'

In [99]:
stem = Path(ip).stem
print(f"Processing: {ip}")

# Load and convert BGR->RGB
image_bgr = cv2.imread(ip)

image = image_bgr[..., ::-1]
# image = deskew_grayscale(image)
h, w = image.shape[:2]

# 1) Layout detection
layout = model.detect(image)  # returns lp.Layout

Processing: data/newspapers/service-ndnp-prru-batch_prru_ballena_ver01-data-sn90070270-00271761995-1899021301-0346.jpg


In [105]:
layout = reclassify_by_text_density(image, layout, ocr_agent, LANGS)

In [109]:
text_blocks  = lp.Layout([b for b in layout if b.type in ("Text", "Title")])
figure_blocks = lp.Layout([b for b in layout if b.type == "Figure"])
text_blocks   = lp.Layout([b for b in text_blocks if not any(b.is_in(fb) for fb in figure_blocks)])


In [114]:
ordered_blocks = group_into_columns(layout, w)


In [118]:
ocrd = extract_text_from_blocks(image, ordered_blocks)

In [119]:
ocrd

[(TextBlock(block=Rectangle(x_1=572.2713623046875, y_1=1250.029052734375, x_2=1743.3974609375, y_2=2069.56494140625), text=None, id=None, type=Text, parent=None, next=None, score=0.8399496078491211),
  'Después de ocho días de publi-\nsados los artículos ; Justicia... . Ca-\ntalana! se descuelga ahora el se-\nior Juez de instrucción con la\nlenuncia y secuestro de nuestras\nedicciones. Si para convencérse el\nseñor Juez que debía procesarnos\nha necesitado ocho dias, no nos ex-\ntraña que los señores Magistrados\nen tres días solamente que emplea-\nron para dictar la sentencia, hicieran\nun adefesto juridico.'),
 (TextBlock(block=Rectangle(x_1=548.9883422851562, y_1=2668.904541015625, x_2=1608.7003173828125, y_2=3341.67138671875), text=None, id=None, type=Text, parent=None, next=None, score=0.9809884428977966),
  'Mal parado deja el señor Agua-\nyo el celo del señor Fiscal; pues no\n\nabiéndonos éste denunciado, nos\ndenuncia el Juez, como diciendo:\nSeñor Fiscal, aquí hay un delito y\

In [120]:
all_text = []
for _, t in ocrd:
    if t:
        all_text.append(t)
out_txt = OUT_TXT_DIR / f"{stem}.txt"
# out_txt.write_text("\n\n".join(all_text), encoding="utf-8")

In [124]:
print('\n\n'.join(all_text))

Después de ocho días de publi-
sados los artículos ; Justicia... . Ca-
talana! se descuelga ahora el se-
ior Juez de instrucción con la
lenuncia y secuestro de nuestras
edicciones. Si para convencérse el
señor Juez que debía procesarnos
ha necesitado ocho dias, no nos ex-
traña que los señores Magistrados
en tres días solamente que emplea-
ron para dictar la sentencia, hicieran
un adefesto juridico.

Mal parado deja el señor Agua-
yo el celo del señor Fiscal; pues no

abiéndonos éste denunciado, nos
denuncia el Juez, como diciendo:
Señor Fiscal, aquí hay un delito y
como usted no a procedido con-
tra él, yo, en uso de mis atribu-
ciones, procedo; perque yo, aunque
usted no lo crea, soy más papista
que el Papa. —

- El Señor Secretario de Justicia
debe tener en cuenta el excesivo
celo del señor Juez de Instruc-
ción de Ponce, para recompensarlo
con un ascenso en la primera opor-
tunidad..

No: es imposible que nos ame-
ricanicemos, mientras llevemos en
la masa de la sangre los morbosis-

In [164]:
layout

Layout(_blocks=[TextBlock(block=Rectangle(x_1=548.3800048828125, y_1=3735.870849609375, x_2=1603.7742919921875, y_2=4021.201904296875), text=None, id=None, type=Text, parent=None, next=None, score=0.9890835881233215), TextBlock(block=Rectangle(x_1=548.9883422851562, y_1=2668.904541015625, x_2=1608.7003173828125, y_2=3341.67138671875), text=None, id=None, type=Text, parent=None, next=None, score=0.9809884428977966), TextBlock(block=Rectangle(x_1=156.83599853515625, y_1=6449.5234375, x_2=5520.720703125, y_2=9826.93359375), text=None, id=None, type=Text, parent=None, next=None, score=0.9686573147773743), TextBlock(block=Rectangle(x_1=541.1197509765625, y_1=4038.9970703125, x_2=1596.431640625, y_2=4653.40625), text=None, id=None, type=Text, parent=None, next=None, score=0.9580318331718445), TextBlock(block=Rectangle(x_1=1619.6480712890625, y_1=5121.2255859375, x_2=2705.80810546875, y_2=5650.86328125), text=None, id=None, type=Text, parent=None, next=None, score=0.9552755355834961), TextBlo

In [165]:
debug_draw_layout(image, lp.Layout([layout[1]]), path=f"outputs/debug.png")

In [127]:
alto_path = 'data/newspapers/0346.xml'

In [130]:
for l in layout:
    break

In [170]:
lines

[{'geom': <POLYGON ((13980 1468, 13980 1568, 13780 1568, 13780 1468, 13980 1468))>,
  'x': 13780.0,
  'y': 1468.0,
  'w': 200.0,
  'h': 100.0,
  'strings': [{'x': 13780.0,
    'y': 1468.0,
    'w': 108.0,
    'h': 100.0,
    'text': 'O',
    'text_norm': 'O',
    'subs_type': None},
   {'x': 13960.0,
    'y': 1476.0,
    'w': 20.0,
    'h': 60.0,
    'text': 'i',
    'text_norm': 'i',
    'subs_type': None}],
  'text': 'O i'},
 {'geom': <POLYGON ((3180 1748, 3180 1848, 3128 1848, 3128 1748, 3180 1748))>,
  'x': 3128.0,
  'y': 1748.0,
  'w': 52.0,
  'h': 100.0,
  'strings': [{'x': 3128.0,
    'y': 1748.0,
    'w': 52.0,
    'h': 100.0,
    'text': 'V',
    'text_norm': 'V',
    'subs_type': None}],
  'text': 'V'},
 {'geom': <POLYGON ((5760 2672, 5760 3616, 3184 3616, 3184 2672, 5760 2672))>,
  'x': 3184.0,
  'y': 2672.0,
  'w': 2576.0,
  'h': 944.0,
  'strings': [{'x': 3184.0,
    'y': 2672.0,
    'w': 2576.0,
    'h': 944.0,
    'text': 'OSÍDCDá',
    'text_norm': 'OSÍDCDá',
    'subs_

In [159]:
lines = parse_alto_textlines(alto_path)
idx, geoms, lines_list = build_line_index(lines)
elems = lp_elements_to_bboxes(layout)

In [167]:
len(elems), len(layout)

(53, 53)

In [139]:
containment='intersect'

In [168]:
results = []
for el in [elems[1]]:
    txt, matched_lines = text_for_lp_region_lines(
        el['bbox'], idx, geoms, lines_list, containment='intersect'
    )
    results.append({
        **el,
        'text': txt,
        'lines': [{'x': ln['x'], 'y': ln['y'], 'w': ln['w'], 'h': ln['h']} for ln in matched_lines]
    })

In [154]:
region = box(*el['bbox'])
candidates = idx.query(region)

In [169]:
results

[{'id': 1,
  'type': 'Text',
  'score': 0.9809884428977966,
  'bbox': (548.9883422851562,
   2668.904541015625,
   1608.7003173828125,
   3341.67138671875),
  'text': '',
  'lines': []}]

In [149]:
text_for_lp_region_lines(el['bbox'], idx, g2, containment=containment)

('', [])

In [141]:
results

[{'id': 0,
  'type': 'Text',
  'score': 0.9890835881233215,
  'bbox': (548.3800048828125,
   3735.870849609375,
   1603.7742919921875,
   4021.201904296875),
  'text': '',
  'lines': []}]